# Week 6 · Day 1 — Multiple Linear Regression (Vectorized) + A Real Project

Last week we built a model with **one** feature: study hours → marks. But real problems have **many** features at once. A house's price depends on its size *and* its bedrooms *and* its age *and* its location — all together.

Today we extend our model to handle many features, and we meet the tool that makes it clean and fast: **vectorization**. Then we put it to work on a **real housing dataset** with scikit-learn.

**The plan:**
1. From one feature to many — the idea of weights and the dot product.
2. Build multiple regression *from scratch* in NumPy (vectorized).
3. A real project: predict house prices with scikit-learn.
4. Interpret the model — which features drive price?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## 1. From one feature to many

Last week's model was a single straight line:

$$ y = w \cdot x + b $$

one weight `w`, one feature `x`. With **several** features, each one gets its own weight:

$$ y = w_1 x_1 + w_2 x_2 + w_3 x_3 + \dots + w_n x_n + b $$

For a house: `price = w₁·(size) + w₂·(bedrooms) + w₃·(age) + ... + b`. Each weight says how much that feature pushes the price up or down.

Writing out that long sum is tedious. Instead we use the **dot product**, which multiplies matching pairs and adds them up in one operation:

$$ y = \mathbf{w} \cdot \mathbf{x} + b $$

where **w** and **x** are now *vectors* (lists of numbers). This is **vectorization** — expressing a whole loop as a single array operation. It's the reason NumPy exists.

In [ ]:
# The dot product, by hand vs NumPy — same answer
weights = np.array([200, 15000, -900])      # price per sqft, per bedroom, per year of age
house   = np.array([2000, 3, 20])           # a 2000 sqft, 3-bed, 20-year-old house

# the long way: multiply pairs and add
long_way = 200*2000 + 15000*3 + (-900)*20
# the vectorized way: one operation
vector_way = np.dot(weights, house)

print("long way   :", long_way)
print("vector way :", vector_way)
print("same answer:", long_way == vector_way)

One `np.dot` replaced three multiplications and two additions. With 50 features and 20,000 houses, that saving is enormous — and it's why every machine-learning library is built on vectorized operations.

---
## 2. Building multiple regression from scratch

We'll load a housing dataset and build the model with pure NumPy, exactly as we did last week — just vectorized to handle many features. This keeps the machinery visible before we hand it to a library.

*(The dataset here is a realistic housing set. In the real Kaggle version — "House Sales in King County" — you'd load `kc_house_data.csv` the same way; every step below is identical.)*

In [ ]:
df = pd.read_csv("house_prices.csv")
print("shape:", df.shape)
df.head()

In [ ]:
# Quick look at what relates to price (Week 4 skills)
df.corr()["price"].drop("price").sort_values(ascending=False).round(3)

`sqft_living`, `bedrooms`, and `bathrooms` correlate most with price — no surprise. Let's use a handful of strong features and build a model that uses **all of them at once**.

In [ ]:
# Pick several features (a matrix X) and the target (y)
features = ["sqft_living", "bedrooms", "bathrooms", "grade"]
X = df[features].values     # a MATRIX: rows = houses, columns = features
y = df["price"].values      # the target

print("X shape:", X.shape, " (rows = houses, columns = features)")
print("y shape:", y.shape)

### One important preparation step: scaling

Our features are on wildly different scales — `sqft_living` is in the thousands, `bedrooms` is a single digit. Gradient descent struggles when features have very different ranges (it takes huge steps for one, tiny for another). So we **standardize** each feature to have mean 0 and a similar spread. This is standard practice, and scikit-learn will do it for us later — but let's do it by hand once to see it.

In [ ]:
# Standardize: (value - mean) / std, done per column
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std

# also scale y so the numbers stay manageable during training
y_mean, y_std = y.mean(), y.std()
y_scaled = (y - y_mean) / y_std

print("feature means after scaling:", np.round(X_scaled.mean(axis=0), 3))  # ~0
print("feature stds  after scaling:", np.round(X_scaled.std(axis=0), 3))   # ~1

### The vectorized model, cost, and gradients

Here is the whole model in vectorized form. Compare it to last week — the structure is identical, but `X @ w` (matrix multiplication) does the prediction for **every house and every feature at once**, with no loops.

In [ ]:
def predict(X, w, b):
    # X @ w is the dot product for every row at once — one operation, all houses
    return X @ w + b

def cost(X, y, w, b):
    errors = predict(X, w, b) - y
    return np.mean(errors ** 2) / 2

def gradients(X, y, w, b):
    errors = predict(X, w, b) - y
    grad_w = (X.T @ errors) / len(y)   # one gradient per feature, vectorized
    grad_b = np.mean(errors)
    return grad_w, grad_b

### Train it

Start all weights at zero and run gradient descent — the same loop as last week. Now `w` is a *vector* of weights (one per feature), all updated together.

In [ ]:
n_features = X_scaled.shape[1]
w = np.zeros(n_features)     # one weight per feature, all starting at 0
b = 0.0
learning_rate = 0.1
history = []

for epoch in range(1, 1001):
    grad_w, grad_b = gradients(X_scaled, y_scaled, w, b)
    w = w - learning_rate * grad_w
    b = b - learning_rate * grad_b
    history.append(cost(X_scaled, y_scaled, w, b))

print("training done.")
print("final cost:", round(history[-1], 4))
print("\nlearned weight for each feature (on scaled data):")
for name, weight in zip(features, w):
    print(f"  {name:12s}: {weight:+.3f}")

In [ ]:
# The cost fell as it learned — same picture as last week, now with 4 features
plt.plot(history, color="purple")
plt.xlabel("epoch"); plt.ylabel("cost")
plt.title("Cost falling during training (multiple regression)")
plt.grid(True, alpha=0.3)
plt.show()

The largest weight went to `sqft_living` — the model learned, on its own, that living area matters most for price. We built a working multi-feature model from scratch. 

But scaling by hand, choosing a learning rate, running a loop — that's a lot of manual work. For real projects, scikit-learn does all of it in a few lines. Now that you know what's happening underneath, let's use it.

---
## 3. The real project: house prices with scikit-learn

This is how machine learning is done in practice. We'll run the full professional workflow: **load → explore → split → train → evaluate → interpret.**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

### Step 1 — Features and target
This time let's use **more** features — one of the advantages of multiple regression is that it happily takes many inputs.

In [ ]:
features = ["sqft_living", "sqft_lot", "bedrooms", "bathrooms",
            "floors", "age", "condition", "grade", "loc_score"]
X = df[features]
y = df["price"]

print(f"Using {len(features)} features to predict price")

### Step 2 — Train/test split (the golden rule)

We hold back 20% of the houses as a **test set** the model never sees during training. We'll judge the model only on those — because a model graded on data it already studied tells us nothing about new houses. This one discipline prevents the most common beginner mistake.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"training houses: {len(X_train)}")
print(f"testing houses : {len(X_test)}  (held back, unseen during training)")

### Step 3 — Train the model (three lines)

Everything we built by hand — the scaling, the gradient descent, the loop — is now inside `.fit()`. This is the payoff of understanding it first: the library is no longer a mystery.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print("model trained.")

### Step 4 — Evaluate on the unseen test set

Two standard regression metrics:
- **R²** — the fraction of price variation the model explains, from 0 (useless) to 1 (perfect).
- **RMSE** — the typical prediction error, in dollars ("off by about \$X on a typical house").

In [ ]:
predictions = model.predict(X_test)

r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"R²   = {r2:.3f}   (explains {r2*100:.0f}% of the variation in price)")
print(f"RMSE = ${rmse:,.0f}   (typical error on a house)")

In [ ]:
# Predicted vs actual — a perfect model would put every point on the red line
plt.scatter(y_test, predictions, alpha=0.3, s=15)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, color="red", linewidth=2, label="perfect prediction")
plt.xlabel("actual price"); plt.ylabel("predicted price")
plt.title("Predicted vs actual house prices (test set)")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

The points cluster tightly around the red line — the model predicts real, unseen house prices well.

---
## 4. Interpreting the model — which features drive price?

A linear model is not just a predictor, it's an *explanation*. Each feature's **coefficient** tells us how much the price changes per unit of that feature (holding the others fixed).

In [ ]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_
}).sort_values("coefficient", ascending=False)

print(coefficients.to_string(index=False))

Read these in plain English — for example, each extra square foot of living space adds about that many dollars to the predicted price, while each year of `age` *subtracts* value (a negative coefficient). The model has recovered sensible, real-world relationships purely from data.

**One honest caveat.** A large coefficient shows *association*, not *causation*. Adding a bedroom won't necessarily raise a house's value by exactly that coefficient — bigger houses simply tend to have more bedrooms *and* higher prices. The model captures patterns; it doesn't prove cause. (Same lesson as correlation in Week 4.)

In [ ]:
# Use the model to price a specific house
new_house = pd.DataFrame([{
    "sqft_living": 2500, "sqft_lot": 5000, "bedrooms": 4, "bathrooms": 2.5,
    "floors": 2, "age": 10, "condition": 3, "grade": 8, "loc_score": 0.5
}])

predicted_price = model.predict(new_house)[0]
print(f"Predicted price for this house: ${predicted_price:,.0f}")

---
## Summary

- **Multiple linear regression** predicts from many features at once: `y = w₁x₁ + ... + wₙxₙ + b`.
- **Vectorization** (the dot product, `X @ w`) expresses the whole calculation as one fast array operation — no loops. It's the foundation of all NumPy-based ML.
- The from-scratch model is the **same gradient descent as Week 5**, just with a vector of weights instead of one.
- **Feature scaling** helps gradient descent converge when features have very different ranges.
- In practice, **scikit-learn** does all of it in a few lines — `train_test_split`, `LinearRegression().fit()`, `.predict()` — and you now know exactly what those lines do.
- Evaluate regression with **R²** (variation explained) and **RMSE** (typical error in real units), always on a held-out **test set**.
- A model's **coefficients** explain which features matter — but association is not causation.

Next time: some relationships aren't straight lines but **curves** — we'll fit them with *polynomial regression*, using this very same machinery.